# Sadeghi et al. (2019): neural claims (CRCNS bst-2)

Paper: Sadeghi M, Zhai X, Stevenson IH, Escabi MA (2019). A neural ensemble correlation code for
sound category identification. PLoS Biol 17(10):e3000449, doi:10.1371/journal.pbio.3000449. The
claims file (claims/sadeghi-2019.yaml) was written from the version of record (PMC6788721) with
its supplement.

Dataset: CRCNS bst-2 (doi:10.6080/K03X84V3), read from the public bucket crcnsarchive at
bst-2/live/data.tar.gz. CRCNS data sets are not versioned, so the version is recorded as
`unversioned` (dataset `crcns:bst-2/unversioned`); the paper cites the data set's DOI and no version, and this is the only copy the
bucket holds (last modified 2026-02-11). The MD5 sums from bst-2/live/checksums.md5 for the files
read: data.tar.gz 916ae724cd3334a16a4396b861e4abe0; docs/crcns_bst-2_data_description.pdf
81457299a36f12a18f2eb8afa5ef1862; docs/FigureData.pdf d0a76ae42a544a60c02912762be2127b. The
data description (version 0.6, 26 August 2019) and its figure list were read first.

This notebook tests the neural claims; the auditory model claims are in
reproduce_sadeghi_2019_model.ipynb.

| Claim | Tested here |
|---|---|
| fig1k-example-temporal-widths | tested |
| fig1-snr-noise-stronger | tested |
| fig1-s5-model-predicts-neural | blocked: data missing (no tone-pip responses, so no best frequencies) |
| fig2-noise-restricted | tested |
| fig3a-example-tonotopy-removal | tested |
| fig3a-example-classifier-order | tested |
| fig3b-population-above-chance | tested |
| figs6-per-sound-above-chance | tested |
| figs9-mixing-coefficient | tested |
| figs10-amua-tmua | blocked: data missing (no raw voltage traces, so no thresholded MUA) |
| fig4-example-original-vs-equalized | tested |
| fig4e-population-original-vs-equalized | tested |
| fig4f-example-equalized-classification | tested |
| fig4h-population-equalized-classification | tested |
| fig8b-example-pca-variance | tested |
| fig8e-example-categorization | tested |
| fig8f-population-categorization | tested |

Method: the correlations port the paper's MATLAB code in the data set
(Code/MUACorrAnalysis/rastermuashuffledxcorrwinfast.m); the classifier follows the Methods
("Noiseless neural classifier", "Single-trial neural classifier") with the readings (1) to (5)
that the claims file names. Every reading is run and printed; the readings are labelled
(1a) each of the 100 random samples is one decision, (1b) the 100 samples' log-likelihoods are
summed for one decision; (2a) the number of PCs chosen on validation accuracy, (2b) on an inner
split of the model data; st_a and st_b are the spectro-temporal classifier with a chosen on the
model or the validation log-likelihood. The aMUA sampling rate is the files' 2034.5 Hz.

In [ ]:
import time, resource
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import sadeghi as S

T_START = time.time()
P1_EXAMPLE, P2_EXAMPLE, P3_EXAMPLE = "P1_Site04", "P2_Site11", "P3_Site04"  # data set's FigureData list (finding aid)
SEED = {"P1": 101, "P2": 202, "P3": 303, "P3pca": 304}  # standard choices; one generator per site, from these

## Loading

One pass over the tar stream. Each neural file is parsed in memory, reduced to its
correlations and classifier results, and released before the next one is read.

In [ ]:
desc, R1, R2, R3, SH, NAMES, CODE = {}, {}, {}, {}, {}, [], {}
def want(n):
    NAMES.append(n)  # every file member of the tar, for the data check
    return ("/NeuralData/Paradigm" in n and n.endswith(".mat")) or n.endswith("/Code/Examples/NeuralExample.m")
for name, raw in S.stream_members(want):
    if name.endswith(".m"):
        CODE[name.split("/")[-1]] = raw.decode("latin-1")
        continue
    d = S.read_site(name, raw)
    del raw
    site = d["name"]
    desc[site] = S.describe(d)
    SH[site] = {key: S.split_half(r) for key, r in d["ras"].items()}  # figs10-amua-tmua, aMUA part
    i = int(site[-2:])
    if site.startswith("P1"):
        R1[site] = S.process_p1(d, np.random.default_rng(SEED["P1"] * 100 + i), noiseless=(site == P1_EXAMPLE))
    elif site.startswith("P2"):
        R2[site] = S.process_p2(d, np.random.default_rng(SEED["P2"] * 100 + i))
    else:
        R3[site] = S.process_p3(d, np.random.default_rng(SEED["P3"] * 100 + i))
        if site == P3_EXAMPLE:
            PCA3 = S.p3_example_pca(d, np.random.default_rng(SEED["P3pca"]))
    del d
    print(f"{site} done at {time.time() - T_START:.0f} s", flush=True)
P1_SITES, P2_SITES, P3_SITES = sorted(R1), sorted(R2), sorted(R3)
print("peak memory (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

## Data check

In [ ]:
tops = {}
for n in NAMES:
    parts = n.split("/")
    tops.setdefault("/".join(parts[:3]) if parts[1] in ("NeuralData", "Sounds", "Code") else "/".join(parts[:2]), []).append(n)
print("tar members (files):", len(NAMES), "|", ", ".join(f"{k} {len(v)}" for k, v in sorted(tops.items())))
print("files other than .mat and .wav:", sorted(n for n in NAMES if not n.endswith((".mat", ".wav"))))
print("fields of every neural file:", sorted({repr(desc[s_]["fields"]) for s_ in desc}))
print("Param fields:", sorted({tuple(sorted(desc[s_]["param"])) for s_ in desc}))
print("files:", len(P1_SITES), "paradigm 1 sites,", len(P2_SITES), "paradigm 2 sites,", len(P3_SITES), "paradigm 3 sites")
fs_all = sorted({f for s in desc.values() for f in s["fs"]})
print("sampling rates in the files (Hz):", fs_all, "; helper uses", S.FS, "; the paper states 2000")
for site in P1_SITES + P2_SITES + P3_SITES:
    s = desc[site]
    conds = s["cond"]
    tr = [c["trials"] for c in conds.values()]
    ns = sorted({c["samples"] for c in conds.values()})
    ok = all(c["trials_equal"] and c["samples_equal"] and c["channels"] == 16 for c in conds.values())
    rng_ = (min(c["min"] for c in conds.values()), max(c["max"] for c in conds.values()))
    clock = np.array([c["clock"] for c in conds.values()])
    print(f"{site} animal {s['param']['AnimalNo']} date {s['param']['RecordDate']} block {s['param']['BlockNo']}"
          f" | conditions {len(conds)} trials {min(tr)}-{max(tr)} samples {ns} ({ns[0] / S.FS:.3f} s)"
          f" | equal across channels {ok} | aMUA range {rng_[0]:.2f} to {rng_[1]:.1f}"
          f" | odd/even PSTH r {np.median(clock[:, 0]):.2f}, shifted 250 ms {np.median(clock[:, 1]):.2f}")
for p, sites in (("P1", P1_SITES), ("P2", P2_SITES), ("P3", P3_SITES)):
    an = {}
    for site in sites:
        an.setdefault(desc[site]["param"]["AnimalNo"], []).append(site)
    print(p, "sites per animal:", {k: len(v) for k, v in an.items()})
    t = [c["trials"] for site in sites for c in desc[site]["cond"].values()]
    print(p, "trials per sound, range", min(t), "to", max(t))
# recordings deposited more than once: identical channel rasters (first two trials hashed)
seen = {}
for site in P1_SITES + P2_SITES + P3_SITES:
    for key, c in desc[site]["cond"].items():
        for ch, h in enumerate(c["hash"]):
            seen.setdefault(h, []).append((site, key, ch))
dups = [v for v in seen.values() if len(v) > 1]
print("identical rasters found:", len(dups), dups[:5])
same_block = {}
for site in P2_SITES + P3_SITES:
    p = desc[site]["param"]
    same_block.setdefault((p["AnimalNo"], p["RecordDate"]), []).append((site, p["BlockNo"]))
print("paradigm 2 and 3 files by animal and date (site, block):", same_block)

The aMUA values are positive envelopes (arbitrary units). Every raster is 6104 samples at
2034.5 Hz, 3.0 s, so the files' rate, not the paper's 2 kHz, gives the stated sound
duration. Responses are locked to the trial start: the PSTHs of odd and even trials
correlate strongly with true alignment and near zero when one is shifted by 250 ms, at
every site and condition (printed above), so the rasters share the clock of the trials.

## Shared measures

In [ ]:
def width_at(curve, tau, frac):
    """Full width (ms) of the contiguous region around the peak where curve >= frac * peak,
    with linear interpolation of the crossings."""
    i0 = int(np.argmax(curve))
    thr = frac * curve[i0]
    def cross(step):
        i = i0
        while 0 <= i + step < len(curve) and curve[i + step] >= thr:
            i += step
        if not (0 <= i + step < len(curve)):
            return tau[i]
        a, b = curve[i], curve[i + step]
        return tau[i] + (tau[i + step] - tau[i]) * (a - thr) / (a - b)
    return cross(1) - cross(-1)


def side_peak(curve, tau, min_lag=10.0):
    """Lag (ms) of the largest local maximum at positive lags beyond min_lag."""
    m = tau > min_lag
    c, t = curve[m], tau[m]
    loc = [i for i in range(1, len(c) - 1) if c[i] > c[i - 1] and c[i] >= c[i + 1]]
    if not loc:
        return np.nan, np.nan
    j = max(loc, key=lambda i: c[i])
    return t[j], c[j]


def within(v, e, tol):
    return bool(abs(v - e) <= tol + 1e-9)


def ttest_chance(x, chance):
    t, p2 = stats.ttest_1samp(x, chance)
    return t, p2, (p2 / 2 if t > 0 else 1 - p2 / 2)


READ = ["1a2a", "1a2b", "1b2a", "1b2b"]
READ_TXT = {"1a2a": "(1a) each sample a decision, (2a) PCs on validation",
            "1a2b": "(1a) each sample a decision, (2b) PCs on an inner split",
            "1b2a": "(1b) 100 samples summed, (2a) PCs on validation",
            "1b2b": "(1b) 100 samples summed, (2b) PCs on an inner split"}
TAU = R1[P1_SITES[0]]["corr"][0]["whole"]["tau_ms"]
DUR = S.DURATIONS_MS
print("durations (ms):", np.round(DUR, 1))

## fig1k-example-temporal-widths

> At the example paradigm 1 site, stimulus-driven temporal correlations are fast for crowd,
> water and fire (3.0, 3.0 and 3.9 ms wide at 50% of maximum; 5.9, 6.9 and 11.8 ms at 10%),
> the bird chorus has a sharp peak (5.9 ms half width) and a slow component (161.2 ms at
> 10%), and the snake has a 7.9 ms half width, 20.6 ms at 10%, and a periodic pattern of about
> 50 ms period at high-frequency sites.

Stimulus-driven (shuffled) temporal correlations c_kk(tau), lags within 100 ms. The lag
step is 0.49 ms at the files' rate. Channel order is not documented in the data set, so
"upper eight channels" is run as channels 9 to 16 and, as a variant, 1 to 8; reading (b) of
the period (best frequency above 2 kHz) needs best frequencies, which the data set lacks.

In [ ]:
code = CODE["NeuralExample.m"].splitlines()
print("NeuralExample.m, the lines that set the window of Fig 1 (line numbers of the file):")
for i, line in enumerate(code, 1):
    if any(k in line for k in ("NS2=", "sA=", "sB=", "RASNMod=", "T0Mod=", "TwMod=", "MaxTau=", "MaxLag=", "rastermuashuffledxcovwinfastN(RASNMod")):
        print(f"   {i:3d}: {line.strip()}")
EXP_W = {("crowd", 0.5): (3.0, 0.5), ("water", 0.5): (3.0, 0.5), ("fire", 0.5): (3.9, 0.5),
         ("crowd", 0.1): (5.9, 0.59), ("water", 0.1): (6.9, 0.69), ("fire", 0.1): (11.8, 1.18),
         ("bird", 0.5): (5.9, 0.59), ("bird", 0.1): (161.2, 16.1), ("snake", 0.5): (7.9, 0.79),
         ("snake", 0.1): (20.6, 2.06)}
ex = R1[P1_EXAMPLE]["corr"]
fig1k = {}
for win in S.WINDOW_READINGS:
    for curve_r in ("mean", "median"):
        vals, hits = {}, 0
        for (snd, frac), (e, tol) in EXP_W.items():
            T = ex[S.P1_SOUNDS.index(snd)][win]["temp"]["sh"]
            if curve_r == "mean":
                v = width_at(T.mean(0), TAU, frac)
            else:
                v = np.median([width_at(T[c], TAU, frac) for c in range(16)])
            vals[(snd, frac)] = v
            hits += within(v, e, tol)
        Ts = ex[4][win]["temp"]["sh"]
        per = {"9-16": side_peak(Ts[8:].mean(0), TAU), "1-8": side_peak(Ts[:8].mean(0), TAU)}
        hits_p = {k: within(p[0], 50, 5) for k, p in per.items()}
        order = vals[("crowd", 0.5)] < vals[("fire", 0.5)] and vals[("water", 0.5)] < vals[("fire", 0.5)]
        fig1k[(win, curve_r)] = (vals, hits, per, hits_p, order)
        print(f"window {win:9s} curve {curve_r:6s}: " + ", ".join(f"{s} {int(f*100)}% {v:.1f}" for (s, f), v in vals.items()))
        print(f"    snake side peak, channels 9-16 {per['9-16'][0]:.1f} ms (height {per['9-16'][1]:.3f}),"
              f" channels 1-8 {per['1-8'][0]:.1f} ms (height {per['1-8'][1]:.3f})")
        print(f"    widths within tolerance {hits}/10; with the period (9-16) {hits + hits_p['9-16']}/11,"
              f" (1-8) {hits + hits_p['1-8']}/11; crowd and water narrower than fire at 50%: {order}")
best = max(fig1k.items(), key=lambda kv: kv[1][1] + max(kv[1][3].values()))
print("best reading:", best[0], "count", best[1][1] + max(best[1][3].values()), "of 11")

fig, ax = plt.subplots(1, 5, figsize=(14, 2.6), sharey=True)
for k, snd in enumerate(S.P1_SOUNDS):
    T = ex[k]["whole"]["temp"]["sh"]
    ax[k].plot(TAU, T.mean(0), "k")
    ax[k].plot(TAU, T[8:].mean(0), "r", lw=0.8)
    ax[k].set_title(snd); ax[k].set_xlabel("lag (ms)")
ax[0].set_ylabel("stimulus-driven corr.")
plt.tight_layout(); plt.show()
print("plotted range:", float(min(ex[k]["whole"]["temp"]["sh"].mean(0).min() for k in range(5))),
      "to", float(max(ex[k]["whole"]["temp"]["sh"].mean(0).max() for k in range(5))))

**Result: under the code's window (the first half of the raster, as NeuralExample.m computes Fig 1) and the channel-averaged curve, all ten widths are within tolerance (crowd 2.8, water 3.1, fire 4.4 ms at 50%; 5.5, 7.1, 12.2 ms at 10%; bird 5.6 and 156.8 ms; snake 7.6 and 20.6 ms), and crowd and water are narrower than fire. The snake side peak is at 60.9 ms, outside 50 +/- 5 ms, so 10 of 11 values match. Under the claims file's two windows (the whole response, 1000 ms windows) 6 to 8 of 11 match; the bird's 10% width there is 138.8 or 50.2 ms.**

## fig1-snr-noise-stronger

> Noise correlations are generally stronger than the stimulus-driven correlations, with
> spectral SNR from -26.0 to -0.9 dB (mean -11.0, SD 7.2) and temporal SNR from -26.1 to -4.1
> dB (mean -6.0, SD 8.2).

SNR = 10 log10 of summed squared stimulus-driven over summed squared noise correlation,
readings (a) all entries, (b) without the zero-lag diagonal; (c) one value per site and
sound, (d) per site averaged over sounds; (e) 13 sites, (f) the example site; and the
window readings.

In [ ]:
def snr_site_sound(c, excl):
    sp_s, sp_n = c["spec"]["sh"].copy(), c["spec"]["noise"].copy()
    te_s, te_n = c["temp"]["sh"].copy(), c["temp"]["noise"].copy()
    if excl:
        off = ~np.eye(16, dtype=bool)
        sp_s, sp_n = sp_s[off], sp_n[off]
        keep = np.abs(TAU) > 1e-9
        te_s, te_n = te_s[:, keep], te_n[:, keep]
    return (10 * np.log10((sp_s ** 2).sum() / (sp_n ** 2).sum()),
            10 * np.log10((te_s ** 2).sum() / (te_n ** 2).sum()))

snr_rows = []
for win in S.WINDOW_READINGS:
    for excl in (False, True):
        V = np.array([[snr_site_sound(R1[s]["corr"][k][win], excl) for k in range(5)] for s in P1_SITES])  # sites, sounds, 2
        for unit in ("c", "d"):
            for rec in ("e", "f"):
                X = V if rec == "e" else V[[P1_SITES.index(P1_EXAMPLE)]]
                X = X.reshape(-1, 2) if unit == "c" else X.mean(1)
                if rec == "f" and unit == "d":
                    continue  # one value only
                sp, te = X[:, 0], X[:, 1]
                hits_m = within(sp.mean(), -11.0, 7.2) + within(te.mean(), -6.0, 8.2)
                hits_b = (within(sp.min(), -26.0, 2.6) + within(sp.max(), -0.9, 0.09)
                          + within(te.min(), -26.1, 2.61) + within(te.max(), -4.1, 0.41))
                neg = ((sp < 0).mean(), (te < 0).mean())
                snr_rows.append((win, "b" if excl else "a", unit, rec, sp, te, hits_m, hits_b, neg))
                print(f"{win:9s} ({'b' if excl else 'a'})({unit})({rec}) n={len(sp):2d} | spectral mean {sp.mean():6.1f} SD {sp.std(ddof=1) if len(sp) > 1 else 0:4.1f}"
                      f" range {sp.min():6.1f} to {sp.max():5.1f} | temporal mean {te.mean():6.1f} SD {te.std(ddof=1) if len(te) > 1 else 0:4.1f}"
                      f" range {te.min():6.1f} to {te.max():5.1f} | means in tol {hits_m}/2, bounds {hits_b}/4,"
                      f" fraction < 0 dB {neg[0]:.2f}, {neg[1]:.2f}")
match = [r[:4] for r in snr_rows if r[6] == 2 and r[7] >= 3]
part = [r[:4] for r in snr_rows if r[6] == 2 and min(r[8]) > 0.5]
print("readings meeting the reproduced rule:", match)
print("readings with both means in tolerance and SNR mostly negative:", part)

**Result: both mean SNRs are within tolerance under most readings (13 sites, one value per site and sound, whole window: spectral -11.8 dB, temporal -8.1 dB), and the SNR is below 0 dB for 97% of spectral and 88% of temporal values. No reading puts three of the four range bounds within tolerance (at most 2 of 4; whole window, per site and sound: spectral -34.0 to 1.0 dB, temporal -29.3 to 7.7 dB). At the example site alone the means are -4.3 and -1.9 dB.**

## fig2-noise-restricted

> Across the 13 paradigm 1 sites, noise correlations are more restricted in frequency and
> time and more consistent across sounds than stimulus-driven correlations, which are broader
> and stimulus dependent (a slow component for the bird chorus and a 20 Hz periodic
> component for the snake).

In [ ]:
def collapse(c, kind):
    M = c["spec"][kind]
    sp = np.array([np.mean(np.concatenate([np.diag(M, o), np.diag(M, -o)])) if o else np.mean(np.diag(M)) for o in range(16)])
    te = c["temp"][kind].mean(0)
    return sp / sp[0], te / te[np.argmin(np.abs(TAU))]

OFF = np.arange(16.0)
fig2 = {}
for win in S.WINDOW_READINGS:
    cur = {kind: np.array([[collapse(R1[s]["corr"][k][win], kind) for k in range(5)] for s in P1_SITES], dtype=object)
           for kind in ("sh", "noise")}
    avg = {kind: (np.mean([[cur[kind][i, k][0] for k in range(5)] for i in range(len(P1_SITES))], 0),
                  np.mean([[cur[kind][i, k][1] for k in range(5)] for i in range(len(P1_SITES))], 0)) for kind in cur}
    res = {}
    for kind in ("sh", "noise"):
        sp, te = avg[kind]  # (sounds, offsets), (sounds, lags)
        res[kind] = {"sp_w": [width_at(np.concatenate([sp[k][::-1], sp[k][1:]]), np.concatenate([-OFF[::-1], OFF[1:]]), 0.5) for k in range(5)],
                     "te_w": [width_at(te[k], TAU, 0.5) for k in range(5)],
                     "sp_far": [sp[k][3:].mean() for k in range(5)],
                     "te_far": [te[k][np.abs(TAU) > 10].mean() for k in range(5)],
                     "sp_sd": sp.std(0, ddof=1).sum(), "te_sd": te.std(0, ddof=1).sum()}
    fig2[win] = (res, avg)
    for meas, lab in (("w", "width at 50% (channels / ms)"), ("far", "mean beyond 2 channels / beyond 10 ms")):
        a_s = res["sh"]["sp_" + meas]; a_n = res["noise"]["sp_" + meas]
        b_s = res["sh"]["te_" + meas]; b_n = res["noise"]["te_" + meas]
        ns_ = sum(n < s for n, s in zip(a_n, a_s)); nt_ = sum(n < s for n, s in zip(b_n, b_s))
        print(f"{win:9s} {lab}: spectral stim {np.round(a_s, 2)} noise {np.round(a_n, 2)} -> noise smaller {ns_}/5;"
              f" temporal stim {np.round(b_s, 2)} noise {np.round(b_n, 2)} -> noise smaller {nt_}/5")
    print(f"{win:9s} SD across sounds summed: spectral stim {res['sh']['sp_sd']:.2f} noise {res['noise']['sp_sd']:.2f};"
          f" temporal stim {res['sh']['te_sd']:.2f} noise {res['noise']['te_sd']:.2f}")
res, avg = fig2["whole"]
print("bird chorus stimulus-driven temporal correlation (whole window) at 50 ms lag:",
      round(float(avg["sh"][1][1][np.argmin(np.abs(TAU - 50))]), 3), "; noise:", round(float(avg["noise"][1][1][np.argmin(np.abs(TAU - 50))]), 3))
print("snake stimulus-driven side peak:", side_peak(avg["sh"][1][4], TAU), "; noise:", side_peak(avg["noise"][1][4], TAU))
fig, ax = plt.subplots(1, 4, figsize=(14, 2.6))
for k, snd in enumerate(S.P1_SOUNDS):
    ax[0].plot(OFF, avg["sh"][0][k], label=snd); ax[1].plot(OFF, avg["noise"][0][k])
    ax[2].plot(TAU, avg["sh"][1][k]); ax[3].plot(TAU, avg["noise"][1][k])
for a, t in zip(ax, ["spectral, stimulus", "spectral, noise", "temporal, stimulus", "temporal, noise"]):
    a.set_title(t)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
print("plotted ranges:", [(round(float(np.min(x)), 2), round(float(np.max(x)), 2)) for x in (avg["sh"][0], avg["noise"][0], avg["sh"][1], avg["noise"][1])])

**Result: in every window reading the noise correlations are narrower than the stimulus-driven ones at 50% of the maximum for 5 of 5 sounds in both domains (whole window, spectral 1.8 to 2.0 against 6.6 to 9.1 channels; temporal 1.2 against 4.8 to 7.5 ms), and their SD across sounds is smaller in both domains (spectral 0.33 against 0.68, temporal 3.63 against 48.67). The bird chorus keeps a stimulus-driven correlation of 0.22 at 50 ms (noise 0.05), and the snake's stimulus-driven curve has a side peak at 55 ms.**

## Single-trial classifier, paradigm 1

Accuracy at each duration, per site, for every reading of (1) and (2); the spectro-temporal
classifier is shown with a chosen on (a) the model or (b) the validation log-likelihood.

In [ ]:
CLF = ["st_a", "st_b", "spec", "temp", "temp_shuf", "rate"]
def acc_table(R, sites, key="single", di=-1):
    return {(r, c): np.array([100 * R[s][key]["acc"][(r, c)][di] for s in sites]) for r in READ for c in CLF}
A1 = acc_table(R1, P1_SITES)
A1_short = acc_table(R1, P1_SITES, di=0)
for r in READ:
    print(READ_TXT[r])
    for c in CLF:
        print(f"   {c:9s} mean at 1 s {A1[(r, c)].mean():5.1f} (SD {A1[(r, c)].std(ddof=1):4.1f}), at 62.5 ms {A1_short[(r, c)].mean():5.1f}")

## fig3a-example-tonotopy-removal

> At the example site, randomizing the channel order of the temporal classifier lowers its
> single-trial accuracy at 1 s from 72% to 35%.

In [ ]:
i_ex = P1_SITES.index(P1_EXAMPLE)
for r in READ:
    t, ts = A1[(r, "temp")][i_ex], A1[(r, "temp_shuf")][i_ex]
    print(f"{READ_TXT[r]}: temporal {t:.1f}% (PCs {R1[P1_EXAMPLE]['single']['pcs'][(r, 'temp')]}),"
          f" without tonotopy {ts:.1f}%; within tolerance: {within(t, 72, 7.2)}, {within(ts, 35, 3.5)}; drop {t - ts:.1f} points")
print("chance 20%")

**Result: removing tonotopy lowers the temporal classifier at 1 s by 20 to 40 points under every reading, but no reading puts both values within tolerance: reading (1a)(2a) gives 78.4% with tonotopy (in tolerance) and 45.1% without (outside 35 +/- 3.5%); reading (1b)(2b) gives 70% and 30%.**

## fig3a-example-classifier-order

> At the example site, the spectro-temporal classifier performs best, followed by the
> spectral and temporal classifiers; the noiseless classifier exceeds the single-trial one
> for the spectral and spectro-temporal classifiers, while for the temporal classifier the
> two are similar.

The noiseless classifier is run at the example site at 1 s only (500 ms shuffled model
windows, 1000 ms shuffled validation windows), which is what the criterion compares.

In [ ]:
NL = R1[P1_EXAMPLE]["noiseless"]
for r in READ:
    for st in ("st_a", "st_b"):
        s_ = {c: A1[(r, c)][i_ex] for c in (st, "spec", "temp")}
        n_ = {c: 100 * NL["acc"][(r, c)][-1] for c in (st, "spec", "temp")}
        order = s_[st] >= s_["spec"] >= s_["temp"]
        nl_ok = (n_[st] - s_[st] >= 5) and (n_["spec"] - s_["spec"] >= 5) and abs(n_["temp"] - s_["temp"]) < 5
        print(f"{READ_TXT[r]}, a ({st[-1]}): single-trial st {s_[st]:.1f} spec {s_['spec']:.1f} temp {s_['temp']:.1f} | "
              f"noiseless st {n_[st]:.1f} spec {n_['spec']:.1f} temp {n_['temp']:.1f} | order {order}, noiseless rule {nl_ok}")

**Result: the single-trial order spectro-temporal >= spectral >= temporal holds under 4 of the 8 reading combinations (for example (1a)(2b) with a on the model data, 86.4, 77.7 and 64.4%), but the noiseless rule holds under none: the noiseless spectral classifier exceeds the single-trial one by 12 to 21 points under the (1a) readings, but the noiseless spectro-temporal classifier is not always 5 points higher, and the noiseless temporal classifier is 13 to 30 points below the single-trial one, not similar.**

## fig3b-population-above-chance

> Across the 13 paradigm 1 sites, all four single-trial classifiers (spectro-temporal,
> spectral, temporal, temporal without tonotopy) perform above chance at 1 s, improve with
> sound duration, and the temporal classifier is degraded substantially when tonotopy is
> removed.

In [ ]:
for r in READ:
    for st in ("st_a", "st_b"):
        line, above = [], 0
        for c in (st, "spec", "temp", "temp_shuf"):
            t, p2, p1 = ttest_chance(A1[(r, c)], 20)
            up = A1[(r, c)].mean() > A1_short[(r, c)].mean()
            above += (p1 < 0.01) and (p2 < 0.01)
            line.append(f"{c} {A1[(r, c)].mean():.1f}% (p1 {p1:.1e}, p2 {p2:.1e}; 62.5 ms {A1_short[(r, c)].mean():.1f})")
        tp = stats.ttest_rel(A1[(r, "temp")], A1[(r, "temp_shuf")])
        drop = (A1[(r, "temp")] - A1[(r, "temp_shuf")]).mean()
        print(f"{READ_TXT[r]}, a ({st[-1]}): " + "; ".join(line))
        print(f"    above chance at p < 0.01 (both sidednesses): {above}/4; tonotopy drop {drop:.1f} points, paired t = {tp.statistic:.2f}, p = {tp.pvalue:.1e}")
by_animal = {}
for s in P1_SITES:
    by_animal.setdefault(desc[s]["param"]["AnimalNo"], []).append(P1_SITES.index(s))
for an, idx in by_animal.items():
    print("animal", an, f"({len(idx)} sites), reading 1a2a, 1 s:", {c: round(float(A1[("1a2a", c)][idx].mean()), 1) for c in ("st_a", "spec", "temp", "temp_shuf")})

fig, ax = plt.subplots(1, 4, figsize=(14, 2.6), sharey=True)
for a, c in zip(ax, ("st_a", "spec", "temp", "temp_shuf")):
    M = np.array([100 * R1[s]["single"]["acc"][("1a2a", c)] for s in P1_SITES])
    a.errorbar(DUR, M.mean(0), M.std(0, ddof=1), color="r"); a.set_xscale("log"); a.axhline(20, color="gray", ls=":")
    a.set_title(c); a.set_xlabel("duration (ms)")
ax[0].set_ylabel("% correct (1a2a)"); plt.tight_layout(); plt.show()

**Result: under every reading, all four single-trial classifiers exceed 20% at 1 s across the 13 sites (reading (1a)(2a): spectro-temporal 77.8%, spectral 76.9%, temporal 68.7%, without tonotopy 43.1%; largest two-sided p 7.8e-07), each is higher at 1 s than at 62.5 ms, and removing tonotopy lowers the temporal classifier by 25.6 points (paired t = 7.87, p = 4.4e-06). Both animals show the same pattern.**

## figs6-per-sound-above-chance

> Single-trial accuracy at 1 s exceeds chance for every sound and classifier except the snake
> sound with the temporal classifiers (p = 0.15 with tonotopy, 0.92 without).

In [ ]:
for r in READ:
    for st in ("st_a", "st_b"):
        n_ok, snake = 0, {}
        rows = []
        for c in (st, "spec", "temp", "temp_shuf"):
            M = np.array([100 * R1[s]["single"]["acc_class"][(r, c)][-1] for s in P1_SITES])  # sites, sounds
            ps = []
            for k, snd in enumerate(S.P1_SOUNDS):
                t, p2, p1 = ttest_chance(M[:, k], 20)
                ps.append((M[:, k].mean(), p2, p1))
                if snd == "snake" and c in ("temp", "temp_shuf"):
                    snake[c] = (M[:, k].mean(), p2, p1)
                else:
                    n_ok += (p2 < 0.05) and (t > 0)
            rows.append(f"{c}: " + ", ".join(f"{snd} {m:.0f}% p={p2:.2g}" for snd, (m, p2, p1) in zip(S.P1_SOUNDS, ps)))
        snake_ns = all(v[1] >= 0.05 or v[0] <= 20 for v in snake.values())
        below = [f"{c} {v[0]:.0f}% p={v[1]:.2g}" for c, v in snake.items() if v[0] < 20 and v[1] < 0.05]
        print(f"{READ_TXT[r]}, a ({st[-1]}): {n_ok}/18 other conditions above chance (two-sided p < 0.05);"
              f" snake temporal not above chance: {snake_ns}; snake significantly below chance (the paper has p = 0.15 and 0.92): {below or 'none'}")
        for row in rows:
            print("     ", row)

**Result: under the (1a) readings, 18 of the 18 non-snake-temporal conditions exceed chance at p < 0.05 with PCs chosen on validation (16 of 18 with the inner split) and the two snake temporal conditions do not. Under the (1b) readings 16 to 18 of 18 exceed chance but a snake temporal condition does too. Under (1a)(2a) the snake temporal classifier without tonotopy is significantly below chance (12%, p = 0.012), where the paper gives p = 0.92.**

## Paradigm 2 classifier

In [ ]:
A2 = {}
for mode in ("eq_3a", "eq_3b", "orig"):
    for di, lab in ((-1, "1s"), (0, "short")):
        A2[(mode, lab)] = acc_table(R2, P2_SITES, key=mode, di=di)
for mode in ("eq_3a", "eq_3b", "orig"):
    print(mode, {r: {c: round(float(A2[(mode, '1s')][(r, c)].mean()), 1) for c in CLF} for r in READ})

## fig4-example-original-vs-equalized

> At the example paradigm 2 site, neural correlations for the original and 1/f
> spectrum-equalized sounds are very similar (Pearson r = 0.92 +/- 0.01 spectral, 0.72 +/-
> 0.07 temporal, over five sounds).

In [ ]:
def r_pair(co, ce, diag):
    if diag:
        a, b = co["spec"]["sh"].ravel(), ce["spec"]["sh"].ravel()
    else:
        off = ~np.eye(16, dtype=bool)
        a, b = co["spec"]["sh"][off], ce["spec"]["sh"][off]
    return np.corrcoef(a, b)[0, 1], np.corrcoef(co["temp"]["sh"].ravel(), ce["temp"]["sh"].ravel())[0, 1]

def se(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

for win in S.WINDOW_READINGS:
    for diag in (True, False):
        V = np.array([r_pair(R2[P2_EXAMPLE]["corr_orig"][k][win], R2[P2_EXAMPLE]["corr_eq"][k][win], diag) for k in range(5)])
        print(f"{win:9s} diagonal {'in ' if diag else 'out'}: spectral r {V[:, 0].mean():.3f} +/- {se(V[:, 0]):.3f},"
              f" temporal r {V[:, 1].mean():.3f} +/- {se(V[:, 1]):.3f} | per sound spectral {np.round(V[:, 0], 2)}, temporal {np.round(V[:, 1], 2)}"
              f" | within tol {within(V[:, 0].mean(), 0.92, 0.01)}, {within(V[:, 1].mean(), 0.72, 0.07)};"
              f" within 10% {within(V[:, 0].mean(), 0.92, 0.092)}, {within(V[:, 1].mean(), 0.72, 0.072)}")
print("temporal r variants, whole window, over the five sounds (verifier round 1 asked whether lag 0 drives it):")
for lab, sel in (("lags other than 0", np.abs(TAU) > 1e-9), ("|lag| > 5 ms", np.abs(TAU) > 5), ("|lag| > 20 ms", np.abs(TAU) > 20)):
    v = [np.corrcoef(R2[P2_EXAMPLE]["corr_orig"][k]["whole"]["temp"]["sh"][:, sel].ravel(),
                     R2[P2_EXAMPLE]["corr_eq"][k]["whole"]["temp"]["sh"][:, sel].ravel())[0, 1] for k in range(5)]
    print(f"   {lab}: r {np.mean(v):.3f} +/- {se(np.array(v)):.3f} (per sound {np.round(v, 2)})")
v = [np.median([np.corrcoef(R2[P2_EXAMPLE]["corr_orig"][k]["whole"]["temp"]["sh"][c], R2[P2_EXAMPLE]["corr_eq"][k]["whole"]["temp"]["sh"][c])[0, 1]
                for c in range(16)]) for k in range(5)]
print(f"   per channel (median over the 16 channels): r {np.mean(v):.3f} +/- {se(np.array(v)):.3f} (per sound {np.round(v, 2)})"
      f" | within 0.72 +/- 0.07: {within(np.mean(v), 0.72, 0.07)}")

**Result: at the example site the spectral correlations of original and 1/f responses agree as reported (whole window, r = 0.930 +/- 0.019 against 0.92 +/- 0.01), but the temporal correlations agree more closely than reported (r = 0.898 +/- 0.022 against 0.72 +/- 0.07), outside the tolerance and outside 10% under every reading. Lag 0 does not drive the temporal value (0.888 without it); without lags within 5 ms it is 0.693.**

## fig4e-population-original-vs-equalized

> Across the 11 paradigm 2 sites, neural correlations for original and 1/f sounds correlate
> at r = 0.95 +/- 0.02 (spectral) and 0.80 +/- 0.02 (temporal) for the same sound, and less
> across different sounds (0.85 +/- 0.02 and 0.57 +/- 0.01).

In [ ]:
EXP4E = [(0.95, 0.02), (0.80, 0.02), (0.85, 0.02), (0.57, 0.01)]
for win in S.WINDOW_READINGS:
    for diag in (True, False):
        same, across = [], []
        for s in P2_SITES:
            co, ce = R2[s]["corr_orig"], R2[s]["corr_eq"]
            same.append([r_pair(co[k][win], ce[k][win], diag) for k in range(5)])
            across.append([r_pair(co[k][win], ce[j][win], diag) for k in range(5) for j in range(5) if j != k])
        same, across = np.array(same), np.array(across)  # sites, pairs, 2
        for avg in ("a", "b"):
            if avg == "a":
                v = [same[:, :, 0].mean(1), same[:, :, 1].mean(1), across[:, :, 0].mean(1), across[:, :, 1].mean(1)]
            else:
                v = [same[:, :, 0].ravel(), same[:, :, 1].ravel(), across[:, :, 0].ravel(), across[:, :, 1].ravel()]
            m = [x.mean() for x in v]
            hits = sum(within(mm, e, t) for mm, (e, t) in zip(m, EXP4E))
            hits10 = sum(within(mm, e, 0.1 * e) for mm, (e, t) in zip(m, EXP4E))
            print(f"{win:9s} diag {'in ' if diag else 'out'} ({avg}): same spectral {m[0]:.3f}+/-{se(v[0]):.3f}, temporal {m[1]:.3f}+/-{se(v[1]):.3f};"
                  f" across spectral {m[2]:.3f}+/-{se(v[2]):.3f}, temporal {m[3]:.3f}+/-{se(v[3]):.3f} | in tol {hits}/4, within 10% {hits10}/4,"
                  f" same > across {m[0] > m[2] and m[1] > m[3]}")

**Result: same-sound correlations exceed across-sound ones in both domains under every reading, but none of the four values is within its stated SE: whole window, same sound spectral 0.879 and temporal 0.917 (reported 0.95 and 0.80), across sounds 0.818 and 0.639 (0.85 and 0.57). At most 3 of the 4 are within 10% (first-half window).**

## fig4f-example-equalized-classification

> At the example paradigm 2 site, classifiers trained on the original sounds and tested on
> the 1/f sounds keep high accuracy (63% spectral, 63% temporal, 71% spectro-temporal, above
> chance), while the spectrum (rate) classifier falls from about 94% on the original sounds to
> chance.

In [ ]:
i2 = P2_SITES.index(P2_EXAMPLE)
for mode in ("eq_3a", "eq_3b"):
    for r in READ:
        for st in ("st_a", "st_b"):
            v = {c: A2[(mode, "1s")][(r, c)][i2] for c in ("spec", "temp", st, "rate")}
            v_orig_rate = A2[("orig", "1s")][(r, "rate")][i2]
            hits = within(v["spec"], 63, 6.3) + within(v["temp"], 63, 6.3) + within(v[st], 71, 7.1) + within(v_orig_rate, 94, 9.4)
            pc = R2[P2_EXAMPLE][mode]["acc_class"]
            tt = {}
            for c in ("spec", "temp", st):
                per_sound = 100 * pc[(r, c)][-1]
                tt[c] = ttest_chance(per_sound, 20)[1]
            pattern = min(v["spec"], v["temp"], v[st]) > 40 and v["rate"] < 30
            print(f"{mode} {r} a({st[-1]}): 1/f spec {v['spec']:.0f} temp {v['temp']:.0f} st {v[st]:.0f} rate {v['rate']:.0f};"
                  f" original rate {v_orig_rate:.0f} | in tol {hits}/4, 1/f rate < 30 {v['rate'] < 30}, pattern {pattern} |"
                  f" t test over 5 sounds p: " + ", ".join(f"{c} {p:.2g}" for c, p in tt.items()))
# reading (b) of the within-site test: over the classification iterations (samples), reading (1a)
for mode in ("eq_3a", "eq_3b"):
    for c in ("spec", "temp"):
        a = A2[(mode, "1s")][("1a2a", c)][i2] / 100
        n = 100 * 5 * (2 if mode == "eq_3a" else 1)
        z = (a - 0.2) / np.sqrt(0.2 * 0.8 / n)
        print(f"{mode} {c}: over {n} samples, accuracy {100 * a:.0f}%, one-sample z against 20% = {z:.1f}")

**Result: at the example site the correlation classifiers trained on the original sounds stay above 40% on the 1/f sounds under the (1a) readings, and the spectrum classifier falls from 86 to 90% on the original sounds to 25 to 31% on the 1/f sounds. With the half split and (1a)(2a), a on the validation data, the four values are within tolerance (spectral 58, temporal 65, spectro-temporal 65, spectrum on original 90%) but the spectrum classifier on 1/f is 31%, not below 30%; with (1a)(2b) it is 29% and three of the four values are within tolerance.**

## fig4h-population-equalized-classification

> Across 11 paradigm 2 sites, the correlation classifiers stay above chance on the 1/f
> sounds at 1 s (spectral 48%, temporal 49%, spectro-temporal 58%), while the spectrum
> classifier exceeds all three correlation classifiers on the original sounds (90.0 +/-
> 2.0%) but falls to 27.0 +/- 1.0% on the 1/f sounds and does not substantially improve with
> duration.

In [ ]:
for mode in ("eq_3a", "eq_3b"):
    for r in READ:
        for st in ("st_a", "st_b"):
            T = A2[(mode, "1s")]
            v = {c: T[(r, c)] for c in ("spec", "temp", st, "rate")}
            vo = A2[("orig", "1s")][(r, "rate")]
            ps = {c: ttest_chance(v[c], 20) for c in ("spec", "temp", st)}
            tests = all(p[1] < 0.01 and p[0] > 0 for p in ps.values())
            hits = (within(v["spec"].mean(), 48, 4.8) + within(v["temp"].mean(), 49, 4.9) + within(v[st].mean(), 58, 5.8)
                    + within(vo.mean(), 90, 2) + within(v["rate"].mean(), 27, 1))
            below = v["rate"].mean() < min(v["spec"].mean(), v["temp"].mean(), v[st].mean())
            rate_short = A2[(mode, "short")][(r, "rate")].mean()
            orig_corr = {c: A2[("orig", "1s")][(r, c)].mean() for c in ("spec", "temp", st)}
            print(f"{mode} {r} a({st[-1]}): 1/f spec {v['spec'].mean():.1f} temp {v['temp'].mean():.1f} st {v[st].mean():.1f}"
                  f" (p2 {', '.join(f'{p[1]:.1e}' for p in ps.values())}) | rate 1/f {v['rate'].mean():.1f}+/-{se(v['rate']):.1f}"
                  f" (62.5 ms {rate_short:.1f}), rate original {vo.mean():.1f}+/-{se(vo):.1f}; original correlation classifiers"
                  f" {', '.join(f'{c} {x:.1f}' for c, x in orig_corr.items())} | tests {tests}, in tol {hits}/5, rate lowest on 1/f {below}")

**Result: across the 11 sites the correlation classifiers stay above chance on the 1/f sounds at 1 s under every reading (half split, (1a)(2a): spectral 50.4, temporal 54.0, spectro-temporal 54.4%, two-sided p < 1e-06), and the spectrum classifier falls from 90.0 +/- 2.2% on the original sounds to 30.0 +/- 1.3% on the 1/f sounds, below every correlation classifier. At most 3 of the 5 values are within tolerance (spectral 50.4%, spectro-temporal 54.4% and the spectrum classifier on the original sounds; temporal 54.0% is just outside 49 +/- 4.9% and the spectrum classifier on 1/f, 30.0%, outside 27 +/- 1%).**

## fig8b-example-pca-variance

> At the example paradigm 3 site, the first two principal components explain 95.6% of the
> variance of the spectral and 85.9% of the temporal neural correlations, and the three
> categories form distinct clusters.

Readings: (a) 500 ms shuffled (stimulus-driven) windows, (b) 1000 ms windows, run with
single-trial correlations and, as a variant, shuffled ones; (e) all six exemplars, (f) the
five-exemplar training sets averaged over folds; spectral features (c) upper triangle with
diagonal, (d) full matrix. 100 windows per exemplar over the whole response.

In [ ]:
for lab in ("a_sh500", "b_st1000", "b_sh1000"):
    for sr in ("c", "d"):
        sp = PCA3[(lab, sr, "spec")]
        te = PCA3[(lab, "c", "temp")]
        for tr in ("ev_all6", "ev_folds"):
            ok = within(sp[tr], 95.6, 9.56) and within(te[tr], 85.9, 8.59)
            print(f"{lab} spectral ({sr}) training set {'(e) six' if tr == 'ev_all6' else '(f) five, folds'}:"
                  f" spectral {sp[tr]:.1f}%, temporal {te[tr]:.1f}% | both in tolerance {ok} |"
                  f" nearest-centroid accuracy in 2 PCs: spectral {sp['nc_acc']:.1f}%, temporal {te['nc_acc']:.1f}% (chance 33.3%)")

**Result: with 1000 ms windows (single-trial correlations) two PCs explain 96.3% of the spectral and 85.3% of the temporal variance, both within tolerance, and a nearest-centroid rule in the two-PC space classifies 88.6% (spectral) and 85.6% (temporal) of held-out exemplar windows correctly (chance 33.3%). With 500 ms shuffled windows the temporal value is 77.0%, just outside 85.9 +/- 8.59%.**

## Paradigm 3 classifier

In [ ]:
A3 = {rd: {} for rd in ("a", "b")}
for rd in ("a", "b"):
    for di, lab in ((-1, "1s"), (0, "short")):
        A3[rd][lab] = {(r, c): np.array([100 * R3[s][rd]["acc"][(r, c)][di] for s in P3_SITES]) for r in READ for c in CLF}
for rd in ("a", "b"):
    for r in READ:
        print(f"reading ({rd}) {r}:", {c: round(float(A3[rd]['1s'][(r, c)].mean()), 1) for c in CLF})

## fig8e-example-categorization

> At the example paradigm 3 site, all correlation classifiers approach 100% correct
> categorization at 1 s, and the spectrum (rate) classifier reaches 94%.

In [ ]:
i3 = P3_SITES.index(P3_EXAMPLE)
for rd in ("a", "b"):
    for r in READ:
        for st in ("st_a", "st_b"):
            v = {c: A3[rd]["1s"][(r, c)][i3] for c in (st, "spec", "temp", "rate")}
            v0 = {c: A3[rd]["short"][(r, c)][i3] for c in (st, "spec", "temp")}
            ok90 = all(v[c] >= 90 for c in (st, "spec", "temp"))
            up = all(v[c] > v0[c] for c in (st, "spec", "temp"))
            print(f"({rd}) {r} a({st[-1]}): 1 s st {v[st]:.0f} spec {v['spec']:.0f} temp {v['temp']:.0f} rate {v['rate']:.0f};"
                  f" 62.5 ms st {v0[st]:.0f} spec {v0['spec']:.0f} temp {v0['temp']:.0f} | all >= 90 {ok90}, rise {up},"
                  f" rate in tol {within(v['rate'], 94, 9.4)}")

**Result: at the example site all three correlation classifiers reach at least 93% at 1 s under every reading (for example reading (a), (1a)(2a): spectro-temporal 95, spectral 93, temporal 98%), each higher than at 62.5 ms (50 to 83%), and the spectrum classifier reaches 91 to 94%.**

## fig8f-population-categorization

> Across 11 paradigm 3 sites, at 1 s the spectro-temporal classifier categorizes 96% of
> trials correctly, the temporal 95%, the spectral 91% and the spectrum (rate) classifier 91%,
> all above chance, and performance improves with duration.

In [ ]:
for rd in ("a", "b"):
    for r in READ:
        for st in ("st_a", "st_b"):
            T = A3[rd]["1s"]
            v = {c: T[(r, c)] for c in (st, "temp", "spec", "rate")}
            exp = {st: (96, 9.6), "temp": (95, 9.5), "spec": (91, 9.1), "rate": (91, 9.1)}
            hits = sum(within(min(v[c].mean(), 100), *exp[c]) or (v[c].mean() >= exp[c][0] - exp[c][1]) for c in v)
            ps = {c: ttest_chance(v[c], 100 / 3) for c in v}
            tests = all(p[1] < 0.01 and p[0] > 0 for p in ps.values())
            up = all(v[c].mean() > A3[rd]["short"][(r, c)].mean() for c in v)
            print(f"({rd}) {r} a({st[-1]}): " + ", ".join(f"{c} {v[c].mean():.1f}+/-{se(v[c]):.1f} (p2 {ps[c][1]:.1e}; 62.5 ms {A3[rd]['short'][(r, c)].mean():.1f})" for c in v)
                  + f" | in tol {hits}/4, tests {tests}, rise {up}")
by_animal = {}
for s in P3_SITES:
    by_animal.setdefault(desc[s]["param"]["AnimalNo"], []).append(P3_SITES.index(s))
for an, idx in by_animal.items():
    print("animal", an, f"({len(idx)} sites), reading (a) 1a2a, 1 s:", {c: round(float(A3["a"]["1s"][("1a2a", c)][idx].mean()), 1) for c in ("st_a", "spec", "temp", "rate")})

**Result: across the 11 sites, under every reading, the four classifiers are within tolerance at 1 s and above 33.3% (reading (a), (1a)(2a): spectro-temporal 94.8, temporal 94.8, spectral 92.2, spectrum 93.3%; two-sided p < 1e-10), and each is higher at 1 s than at 62.5 ms. Both animals show the same values.**

## figs9-mixing-coefficient

> The optimal spectro-temporal classifier weights spectral and temporal cues about equally,
> with a median mixing coefficient of 0.6 (quartiles 0.5 and 0.7) for paradigms 1 and 2 and
> 0.5 (25th percentile 0.4) for paradigm 3.

a in 0 to 1 in steps of 0.1, per site, chosen to maximise the log-posterior of the true
sound summed over (a) the model data or (b) the validation data. Paradigm 2 uses its
original-to-1/f classifier, reading (3)(a); paradigm 3 its readings (a) and (b).

In [ ]:
for r in READ:
  for m2 in ("eq_3a", "eq_3b"):
    for asel in ("a", "b"):
        a1 = np.array([R1[s]["single"]["alpha"][(r, asel)] for s in P1_SITES])
        a2 = np.array([R2[s][m2]["alpha"][(r, asel)] for s in P2_SITES])
        pooled = np.concatenate([a1, a2])
        line = (f"{r} P2 ({m2[-2:]}) a on ({asel}) data: P1 median {np.median(a1):.2f}, P2 median {np.median(a2):.2f},"
                f" pooled 24 sites median {np.median(pooled):.2f} (quartiles {np.percentile(pooled, 25):.2f}, {np.percentile(pooled, 75):.2f})")
        for rd in ("a", "b"):
            a3 = np.array([R3[s][rd]["alpha"][(r, asel)] for s in P3_SITES])
            ok_c = within(np.median(pooled), 0.6, 0.1) and within(np.median(a3), 0.5, 0.1)
            ok_d = within(np.median(a1), 0.6, 0.1) and within(np.median(a2), 0.6, 0.1) and within(np.median(a3), 0.5, 0.1)
            line += (f" | P3 ({rd}) median {np.median(a3):.2f} (25th pct {np.percentile(a3, 25):.2f});"
                     f" pooled rule {ok_c}, per-paradigm rule {ok_d}")
        print(line)
print("per-site a, (1b)(2a), a on the validation data (P1 | P2 (3a) | P2 (3b) | P3 (a)):")
print("   ", [R1[s]["single"]["alpha"][("1b2a", "b")] for s in P1_SITES], "|", [R2[s]["eq_3a"]["alpha"][("1b2a", "b")] for s in P2_SITES], "|",
      [R2[s]["eq_3b"]["alpha"][("1b2a", "b")] for s in P2_SITES], "|", [R3[s]["a"]["alpha"][("1b2a", "b")] for s in P3_SITES])

**Result: the medians of a depend strongly on how a is chosen. One reading puts both medians within tolerance: (1b)(2a) with a chosen on the validation data, pooled 24 sites median 0.70 (quartiles 0.40, 0.90) and paradigm 3 (whole response) median 0.50. With a chosen on the model data, the paradigm 1 and 2 medians are 0.50 to 0.60 but the paradigm 3 medians are 0.80 to 0.90; with a chosen on the validation data and the (1a) readings, the paradigm 3 medians are 0.00 to 0.40. With paradigm 2 a from the whole-response classifier (3b), two combinations pass with a chosen on the validation data ((1a)(2a): pooled 0.70, paradigm 3 0.40, also per paradigm; (1b)(2a): pooled 0.65, paradigm 3 0.50), and the per-site values spread from 0.0 to 1.0.**

## figs10-amua-tmua (aMUA part)

> Correlations from thresholded MUA resemble those from aMUA (median r 0.81 spectral, 0.35
> temporal) but are weaker and noisier; split-half reliability is higher for aMUA (0.98 and 0.83)
> than for tMUA (0.77 and 0.27).

The tMUA needs the raw voltage traces, which the data set does not hold (the data check lists
every member of the tar and every field of the neural files), so four of the six values cannot
be computed and the claim stays blocked. The two aMUA split-half values can: Pearson r between
the stimulus-driven correlations of two halves of the data (S10 Fig F legend), per raster.
Halves are read as (time) the first and second 1.5 s, as NeuralExample.m splits the raster, or
(trial) odd and even trials; sites and sounds as the claims file's readings (a) paradigm 1, (b)
all paradigms. The spectral r is shown with and without the zero-lag diagonal.

In [ ]:
for half in ("time", "trial"):
    for rd, sites in (("a", P1_SITES), ("b", P1_SITES + P2_SITES + P3_SITES)):
        V = np.array([SH[s][key][half] for s in sites for key in SH[s]])
        print(f"halves ({half}), reading ({rd}), {len(V)} site and sound rasters: median r spectral {np.median(V[:, 0]):.3f}"
              f" (off the diagonal {np.median(V[:, 1]):.3f}), temporal {np.median(V[:, 2]):.3f}"
              f" | range spectral {V[:, 0].min():.2f} to {V[:, 0].max():.2f}, temporal {V[:, 2].min():.2f} to {V[:, 2].max():.2f}"
              f" | within 0.98 +/- 0.098: {within(np.median(V[:, 0]), 0.98, 0.098)}, 0.83 +/- 0.083: {within(np.median(V[:, 2]), 0.83, 0.083)}")
for p_, sites in (("P1", P1_SITES), ("P2", P2_SITES), ("P3", P3_SITES)):
    V = np.array([SH[s][key]["time"] for s in sites for key in SH[s]])
    print(f"   {p_}, time halves: median spectral {np.median(V[:, 0]):.3f}, temporal {np.median(V[:, 2]):.3f}")

**Result: the aMUA split-half reliability matches the paper: with the time halves of NeuralExample.m the median r between halves is 0.980 for spectral and 0.830 for temporal correlations over the 65 paradigm 1 rasters (0.980 and 0.842 over all 373), against 0.98 and 0.83; with odd and even trials as halves it is higher (0.996 and 0.986). The tMUA values cannot be computed from the data set, so the claim stays blocked.**

## Summary

| Claim | Outcome |
|---|---|
| fig1k-example-temporal-widths | reproduced (code's window; 6 to 8 of 11 under the claims file's windows) |
| fig1-snr-noise-stronger | partially reproduced (means hold, range bounds do not) |
| fig1-s5-model-predicts-neural | blocked, data missing |
| fig2-noise-restricted | reproduced |
| fig3a-example-tonotopy-removal | partially reproduced |
| fig3a-example-classifier-order | partially reproduced |
| fig3b-population-above-chance | reproduced |
| figs6-per-sound-above-chance | reproduced |
| figs9-mixing-coefficient | reproduced under one reading of many, at the tolerance edge (fragile) |
| figs10-amua-tmua | blocked, data missing (tMUA); the two aMUA split-half values match |
| fig4-example-original-vs-equalized | not reproduced |
| fig4e-population-original-vs-equalized | not reproduced |
| fig4f-example-equalized-classification | partially reproduced |
| fig4h-population-equalized-classification | partially reproduced |
| fig8b-example-pca-variance | reproduced |
| fig8e-example-categorization | reproduced |
| fig8f-population-categorization | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the aMUA rate is the files' 2034.5 Hz, not 2 kHz;
the classifier's open details (how the 100 time samples enter a decision, how the PCs are
chosen, how a is fitted, how paradigm 2 and 3 data are split) are run under every reading the
claims file names, as printed above; the noiseless classifier is run at the example site and
at 1 s only; the spectrum (rate) classifier uses the mean aMUA of each channel in the window.

What was found about the dataset: the files hold aMUA rasters only, with no tone-pip responses
(best frequencies) and no raw voltage (thresholded MUA); the sampling rate is 2034.5 Hz where the
paper states 2 kHz; the channel order along the probe (which channels are high frequency) is not
documented; one paradigm 2 site (P2_Site08) has 13 trials where the paper states 14; the site,
animal and trial counts otherwise match the paper (13 sites, 4 and 9 per animal; 11 and 11, 3
and 8), no raster is deposited twice, and every raster is locked to the trial start.

In [ ]:
print(f"total run time {(time.time() - T_START) / 60:.1f} min")